In [1]:
# 0. KHỞI TẠO MÔI TRƯỜNG LOCAL
import sys
import os
print("Đang chạy ở môi trường local máy tính.")


Đang chạy ở môi trường local máy tính.


In [2]:
import re
import unicodedata
from datasets import load_dataset
import pandas as pd

# 1. TẢI DỮ LIỆU LOCAL
print("Đang tải bộ dữ liệu...")

try:
    # Ưu tiên đọc file local từ thư mục data/raw/
    print("Thử đọc từ thư mục local data/raw/...")
    train_df = pd.read_excel("../data/raw/Train_15k.xlsx")
    val_df = pd.read_excel("../data/raw/Vadation_1,5k.xlsx")
    test_df = pd.read_excel("../data/raw/Test_1k.xlsx")
    print("Tải dữ liệu thành công từ thư mục local data/raw/!")
except FileNotFoundError:
    # Nếu không thấy file local, thử tải trực tiếp từ internet qua ID Drive làm backup
    print("Không tìm thấy file ở data/raw/. Thử tải trực tiếp qua link Google Drive...")
    file_id_train = "1AaEliMj1x9020cqkw_LxUUYgii2pUo_e"
    file_id_test = "1qiTV9iGmm0RQaZohV3mvamzuluM_4t6N"
    train_url = f"https://drive.google.com/uc?export=download&id={file_id_train}"
    test_url = f"https://drive.google.com/uc?export=download&id={file_id_test}"
    train_df = pd.read_excel(train_url)
    test_df = pd.read_excel(test_url)
    val_df = test_df.copy() # fallback
    print("Tải dữ liệu từ URL thành công!")

print(f"Train: {len(train_df):,} mẫu | Validation: {len(val_df):,} mẫu | Test: {len(test_df):,} mẫu")
print(f"Cột: {list(train_df.columns)}")
print()


c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang tải bộ dữ liệu...
Thử đọc từ thư mục local data/raw/...
Tải dữ liệu thành công từ thư mục local data/raw/!
Train: 15,000 mẫu | Validation: 1,500 mẫu | Test: 1,000 mẫu
Cột: ['content', 'summary']



**1. TIỀN XỬ LÝ DỮ LIỆU**

1.1. Làm sạch dữ liệu

In [3]:
# 2. KHẢO SÁT SƠ BỘ

def explore(df, split_name):
    print(f"=== Khảo sát [{split_name}] ===")
    print(f"  Tổng mẫu       : {len(df):,}")
    print(f"  Null 'content' : {df['content'].isna().sum()}")
    print(f"  Null 'summary' : {df['summary'].isna().sum()}")
    print(f"  summary rỗng   : {(df['summary'].str.strip() == '').sum()}")
    print(f"  content rỗng   : {(df['content'].str.strip() == '').sum()}")
    print(f"  Độ dài content (ký tự): min={df['content'].str.len().min()}, "
          f"max={df['content'].str.len().max()}, "
          f"mean={df['content'].str.len().mean():.0f}")
    print(f"  Độ dài summary (ký tự): min={df['summary'].str.len().min()}, "
          f"max={df['summary'].str.len().max()}, "
          f"mean={df['summary'].str.len().mean():.0f}")
    print()

explore(train_df, "train")
explore(val_df,   "validation")
explore(test_df,  "test")


=== Khảo sát [train] ===
  Tổng mẫu       : 15,000
  Null 'content' : 0
  Null 'summary' : 231
  summary rỗng   : 0
  content rỗng   : 0
  Độ dài content (ký tự): min=62, max=5080, mean=1978
  Độ dài summary (ký tự): min=11.0, max=2050.0, mean=523

=== Khảo sát [validation] ===
  Tổng mẫu       : 1,500
  Null 'content' : 0
  Null 'summary' : 22
  summary rỗng   : 0
  content rỗng   : 0
  Độ dài content (ký tự): min=7, max=4745, mean=1945
  Độ dài summary (ký tự): min=7.0, max=1922.0, mean=518

=== Khảo sát [test] ===
  Tổng mẫu       : 1,000
  Null 'content' : 0
  Null 'summary' : 18
  summary rỗng   : 0
  content rỗng   : 0
  Độ dài content (ký tự): min=176, max=4683, mean=2013
  Độ dài summary (ký tự): min=39.0, max=1663.0, mean=525



In [4]:
# 3. CÁC HÀM LÀM SẠCH VĂN BẢN (Được import từ module src.preprocess)
import sys
import os
# Thêm thư mục gốc vào path để có thể import từ src
sys.path.append(os.path.abspath('..'))
from src.preprocess import clean_text, filter_df


In [5]:
# 4. ÁP DỤNG LÀM SẠCH

print("Đang làm sạch dữ liệu train...")
train_df["content"] = train_df["content"].astype(str).apply(clean_text)
train_df["summary"] = train_df["summary"].astype(str).apply(clean_text)

print("Đang làm sạch dữ liệu validation...")
val_df["content"] = val_df["content"].astype(str).apply(clean_text)
val_df["summary"] = val_df["summary"].astype(str).apply(clean_text)

print("Đang làm sạch dữ liệu test...")
test_df["content"] = test_df["content"].astype(str).apply(clean_text)
test_df["summary"] = test_df["summary"].astype(str).apply(clean_text)


Đang làm sạch dữ liệu train...
Đang làm sạch dữ liệu validation...
Đang làm sạch dữ liệu test...


In [6]:
# 5. LỌC BỎ MẪU KHÔNG HỢP LỆ

# Ngưỡng độ dài (ký tự)
MIN_CONTENT_LEN = 100    # bài báo quá ngắn → không đủ thông tin
MAX_CONTENT_LEN = 6000   # giữ phù hợp với max_length của BARTpho (~1024 token ≈ 4000-6000 ký tự)
MIN_SUMMARY_LEN = 30     # tóm tắt quá ngắn → vô nghĩa
MAX_SUMMARY_LEN = 1000   # tóm tắt dài bất thường

def filter_df(df):
    n0 = len(df)
    # Xóa null / rỗng
    df = df.dropna(subset=["content", "summary"])
    df = df[df["content"].str.strip() != ""]
    df = df[df["summary"].str.strip() != ""]

    # Lọc theo độ dài
    c_len = df["content"].str.len()
    s_len = df["summary"].str.len()
    df = df[
        (c_len >= MIN_CONTENT_LEN) & (c_len <= MAX_CONTENT_LEN) &
        (s_len >= MIN_SUMMARY_LEN) & (s_len <= MAX_SUMMARY_LEN)
    ]

    # Loại bỏ trùng lặp (content giống hệt nhau)
    df = df.drop_duplicates(subset=["content"])

    # Loại bỏ mẫu có summary dài hơn / bằng content (tóm tắt không hợp lý)
    df = df[df["summary"].str.len() < df["content"].str.len()]

    print(f"  Trước lọc: {n0:,} | Sau lọc: {len(df):,} "
          f"| Đã loại: {n0 - len(df):,} ({(n0-len(df))/n0*100:.1f}%)")
    return df.reset_index(drop=True)

print("\nLọc dữ liệu train...")
train_clean = filter_df(train_df)
print("Lọc dữ liệu validation...")
val_clean   = filter_df(val_df)
print("Lọc dữ liệu test...")
test_clean  = filter_df(test_df)



Lọc dữ liệu train...
  Trước lọc: 15,000 | Sau lọc: 14,129 | Đã loại: 871 (5.8%)
Lọc dữ liệu validation...
  Trước lọc: 1,500 | Sau lọc: 1,422 | Đã loại: 78 (5.2%)
Lọc dữ liệu test...
  Trước lọc: 1,000 | Sau lọc: 936 | Đã loại: 64 (6.4%)


In [7]:
# 6. THỐNG KÊ SAU LÀM SẠCH

print("\n=== Thống kê sau làm sạch ===")
explore(train_clean, "train_clean")
explore(val_clean,   "val_clean")
explore(test_clean,  "test_clean")



=== Thống kê sau làm sạch ===
=== Khảo sát [train_clean] ===
  Tổng mẫu       : 14,129
  Null 'content' : 0
  Null 'summary' : 0
  summary rỗng   : 0
  content rỗng   : 0
  Độ dài content (ký tự): min=106, max=5075, mean=1904
  Độ dài summary (ký tự): min=34, max=999, mean=501

=== Khảo sát [val_clean] ===
  Tổng mẫu       : 1,422
  Null 'content' : 0
  Null 'summary' : 0
  summary rỗng   : 0
  content rỗng   : 0
  Độ dài content (ký tự): min=142, max=4697, mean=1874
  Độ dài summary (ký tự): min=74, max=979, mean=496

=== Khảo sát [test_clean] ===
  Tổng mẫu       : 936
  Null 'content' : 0
  Null 'summary' : 0
  summary rỗng   : 0
  content rỗng   : 0
  Độ dài content (ký tự): min=175, max=4583, mean=1922
  Độ dài summary (ký tự): min=39, max=997, mean=501



In [8]:
# 7. LƯU KẾT QUẢ
import os
os.makedirs("../data/processed", exist_ok=True)
train_clean.to_excel("../data/processed/train_clean.xlsx", index=False)
val_clean.to_excel("../data/processed/val_clean.xlsx", index=False)
test_clean.to_excel("../data/processed/test_clean.xlsx", index=False)
print("Đã lưu kết quả vào thư mục: ../data/processed/train_clean.xlsx, ../data/processed/val_clean.xlsx và ../data/processed/test_clean.xlsx")


Đã lưu kết quả vào thư mục: ../data/processed/train_clean.xlsx, ../data/processed/val_clean.xlsx và ../data/processed/test_clean.xlsx
